# Data preprocessing

This notebook applies conservative, model-agnostic preprocessing. Raw TSV files are preserved, all label columns are retained, and the official split boundaries are not changed.

In [1]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'data').exists():
    ROOT = Path.cwd().parent
DATA_DIR = ROOT / 'data'
sys.path.insert(0, str(ROOT / 'src' / 'data'))
from preprocess_dataset import write_comment_summary, write_processed_splits


In [2]:
split_ids = write_processed_splits()
write_comment_summary(split_ids)
print('Processed outputs written to', DATA_DIR / 'processed')

Processed outputs written to c:\Users\Siddharth\OneDrive\Desktop\NNDL Project\data\processed


In [3]:
processed_dir = DATA_DIR / 'processed'
for split in ['train', 'validation', 'test']:
    frame = pd.read_csv(processed_dir / f'{split}.csv')
    print(split, frame.shape)
    display(frame.head(2))

comment_summary = pd.read_csv(processed_dir / 'comment_summary.csv')
print('comment summary:', comment_summary.shape)
comment_summary.head()

train (564000, 21)


,id,clean_title,title,text_char_count,text_word_count,image_url,image_host,image_url_valid,image_available,linked_submission_id,...,domain,subreddit,created_utc,created_at,num_comments,score,upvote_ratio,2_way_label,3_way_label,6_way_label
0,awxhir,my walgreens offbrand mucinex was engraved wit...,My Walgreens offbrand Mucinex was engraved wit...,92,15,https://external-preview.redd.it/WylDbZrnbvZdB...,external-preview.redd.it,True,True,NaN,...,i.imgur.com,mildlyinteresting,1.551641e+09,2019-03-03T19:27:24Z,2.0,12,0.84,1,0,0
1,98pbid,this concerned sink with a tiny hat,This concerned sink with a tiny hat,35,7,https://preview.redd.it/wsfx0gp0f5h11.jpg?widt...,preview.redd.it,True,True,NaN,...,i.redd.it,pareidolia,1.534727e+09,2018-08-20T01:10:13Z,2.0,119,0.99,0,2,2


validation (59342, 21)


,id,clean_title,title,text_char_count,text_word_count,image_url,image_host,image_url_valid,image_available,linked_submission_id,...,domain,subreddit,created_utc,created_at,num_comments,score,upvote_ratio,2_way_label,3_way_label,6_way_label
0,cypw96,my xbox controller says hi,My Xbox controller says hi,26,5,https://preview.redd.it/l0ga0tug17k31.jpg?widt...,preview.redd.it,True,True,NaN,...,i.redd.it,mildlyinteresting,1.567436e+09,2019-09-02T14:47:48Z,4.0,25,0.72,1,0,0
1,d0bzlq,new image from the mandalorian,PsBattle: New image from The Mandalorian,30,5,https://external-preview.redd.it/VX7bXDu9Gl8UZ...,external-preview.redd.it,True,True,NaN,...,i.imgur.com,photoshopbattles,1.567745e+09,2019-09-06T04:43:01Z,5.0,21,0.92,1,0,0


test (59319, 21)


,id,clean_title,title,text_char_count,text_word_count,image_url,image_host,image_url_valid,image_available,linked_submission_id,...,domain,subreddit,created_utc,created_at,num_comments,score,upvote_ratio,2_way_label,3_way_label,6_way_label
0,cozywbv,stargazer,stargazer,9,1,http://i.imgur.com/BruWKDi.jpg,i.imgur.com,True,True,2xct9d,...,NaN,psbattle_artwork,1.425139e+09,2015-02-28T15:51:00Z,NaN,3,NaN,0,2,4
1,ctk61yw,yeah,yeah,4,1,http://i.imgur.com/JRZT727.jpg,i.imgur.com,True,True,3f0h7o,...,NaN,psbattle_artwork,1.438173e+09,2015-07-29T12:29:55Z,NaN,2,NaN,0,2,4


comment summary: (353300, 4)


,id,comment_count,top_level_comment_count,max_comment_ups
0,10012t,3,2,3.0
1,10024l,1,1,1.0
2,1002bi,4,4,2.0
3,10076s,7,5,5.0
4,10092l,9,8,104.0


In [4]:
train = pd.read_csv(processed_dir / 'train.csv')
quality_checks = {
'blank_clean_title': int(train['clean_title'].fillna('').str.strip().eq('').sum()),
'invalid_image_urls': int((~train['image_url_valid']).sum()),
'missing_num_comments': int(train['num_comments'].isna().sum()),
'missing_upvote_ratio': int(train['upvote_ratio'].isna().sum()),
'duplicate_ids': int(train['id'].duplicated().sum()),
'missing_labels': int(train[['2_way_label', '3_way_label', '6_way_label']].isna().any(axis=1).sum()),
}
pd.Series(quality_checks, name='count')

blank_clean_title            0
invalid_image_urls        1534
missing_num_comments    167857
missing_upvote_ratio    167857
duplicate_ids                0
missing_labels               0
Name: count, dtype: int64

In [5]:
from pathlib import Path
import pandas as pd

processed_dir = Path("../data/processed")  # use Path("data/processed") if running from project root

for split in ["train", "validation", "test"]:
    df = pd.read_csv(processed_dir / f"{split}.csv")

    print(f"\n===== {split.upper()} =====")
    print("Shape:", df.shape)

    print("\nMissing values:")
    print(df.isna().sum().sort_values(ascending=False).head(10))

    print("\nImage availability:")
    print(df["image_available"].value_counts(dropna=False))

    print("\nLabel distributions:")
    for label in ["2_way_label", "3_way_label", "6_way_label"]:
        print(f"\n{label}")
        print(df[label].value_counts(normalize=True).sort_index().round(4))

    print("\nText statistics:")
    print(df[["text_char_count", "text_word_count"]].describe().round(2))

    print("\nNumeric metadata:")
    print(df[["num_comments", "score", "upvote_ratio"]].describe().round(2))


===== TRAIN =====
Shape: (564000, 21)

Missing values:
linked_submission_id    396143
num_comments            167857
upvote_ratio            167857
domain                  167857
author                   28710
image_host                1534
image_url                 1534
text_word_count              0
clean_title                  0
id                           0
dtype: int64

Image availability:
image_available
True     562466
False      1534
Name: count, dtype: int64

Label distributions:

2_way_label
2_way_label
0    0.6062
1    0.3938
Name: proportion, dtype: float64

3_way_label
3_way_label
0    0.3938
1    0.0246
2    0.5817
Name: proportion, dtype: float64

6_way_label
6_way_label
0    0.3938
1    0.0594
2    0.1901
3    0.0209
4    0.2976
5    0.0383
Name: proportion, dtype: float64

Text statistics:
       text_char_count  text_word_count
count        564000.00        564000.00
mean             41.93             7.46
std              31.80             5.64
min               1.

In [6]:
comments = pd.read_csv(processed_dir / "comment_summary.csv")

print(comments.shape)
print(comments.describe().round(2))
print("Submissions with comments:", len(comments))

(353300, 4)
       comment_count  top_level_comment_count  max_comment_ups
count      353300.00                353300.00        353300.00
mean           17.86                     6.77            98.06
std            57.60                    17.08           705.30
min             1.00                     1.00           -17.00
25%             2.00                     1.00             2.00
50%             4.00                     3.00             3.00
75%            10.00                     6.00            10.00
max           500.00                   458.00         42573.00
Submissions with comments: 353300


## Preprocessing policy

- Normalize Unicode and repeated whitespace in `title` and `clean_title`.
- Preserve punctuation and original wording.
- Convert numeric metadata and timestamps to usable numeric/datetime fields.
- Add text-length, image-validity, image-host, and image-availability fields.
- Preserve missing metadata as missing values rather than converting it to zero.
- Keep all 2-way, 3-way, and 6-way labels.
- Keep the official dataset splits unchanged.
- Aggregate only comment counts for linked submissions; do not duplicate the full comments file into every split.